In [27]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [28]:
df = pd.read_parquet('../data/all_stocks_data.parquet', engine='fastparquet')

In [29]:
df.shape

(1898322, 7)

In [30]:
df.head()

,Date,Open,High,Low,Close,Volume,Stock
0,2010-01-04,43.193587,43.380729,42.975252,43.157200,3640265.0,MMM
1,2010-01-05,43.042836,43.266370,42.471012,42.886887,3405012.0,MMM
2,2010-01-06,43.604279,43.978564,43.411938,43.495110,6301126.0,MMM
3,2010-01-07,43.313158,43.541891,42.689351,43.526295,5346240.0,MMM
4,2010-01-08,43.505481,43.832981,43.302743,43.832981,4073337.0,MMM


In [31]:
print(df.columns)

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Stock'], dtype='object')


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1898322 entries, 0 to 1898321
Data columns (total 7 columns):
 #   Column  Dtype         
---  ------  -----         
 0   Date    datetime64[ns]
 1   Open    float64       
 2   High    float64       
 3   Low     float64       
 4   Close   float64       
 5   Volume  float64       
 6   Stock   object        
dtypes: datetime64[ns](1), float64(5), object(1)
memory usage: 101.4+ MB


In [33]:
df.head()

,Date,Open,High,Low,Close,Volume,Stock
0,2010-01-04,43.193587,43.380729,42.975252,43.157200,3640265.0,MMM
1,2010-01-05,43.042836,43.266370,42.471012,42.886887,3405012.0,MMM
2,2010-01-06,43.604279,43.978564,43.411938,43.495110,6301126.0,MMM
3,2010-01-07,43.313158,43.541891,42.689351,43.526295,5346240.0,MMM
4,2010-01-08,43.505481,43.832981,43.302743,43.832981,4073337.0,MMM


In [34]:
df.isna().sum()

Date           0
Open      110871
High      110871
Low       110871
Close     110871
Volume    110871
Stock          0
dtype: int64

In [35]:
print(df.isna().sum())

Date           0
Open      110871
High      110871
Low       110871
Close     110871
Volume    110871
Stock          0
dtype: int64


In [36]:
# Flag missing Close values
df['is_missing'] = df['Close'].isna().astype(int)

def find_missing_stretches(group):
    group['gap_id'] = (group['is_missing'].ne(group['is_missing'].shift())).cumsum()
    
    missing_stretches = (
        group[group['is_missing'] == 1]
        .groupby('gap_id')
        .agg(
            Stock=('Stock', 'first'),
            start_date=('Date', 'min'),
            end_date=('Date', 'max'),
            missing_days=('Date', 'count')
        )
        .reset_index(drop=True)
    )
    return missing_stretches

# Apply per Ticker
missing_summary = df.groupby('Stock', group_keys=False).apply(find_missing_stretches)

# Sort by longest gaps
missing_summary = missing_summary.sort_values('missing_days', ascending=False)

# Show summary statistics
print("Top 10 longest missing data stretches:")
print(missing_summary)

print("\nSummary of missing streaks by length:")
print(missing_summary['missing_days'].describe())

# (Optional) Count how many tickers have long missing periods
long_gaps = missing_summary[missing_summary['missing_days'] > 10]
tickers_with_long_gaps = long_gaps['Stock'].nunique()
print(f"\nNumber of tickers with gaps > 10 days: {tickers_with_long_gaps}")

Top 10 longest missing data stretches:
   Stock start_date   end_date  missing_days
0    GEV 2010-01-04 2024-03-26          3581
0   SOLV 2010-01-04 2024-03-25          3580
0   VLTO 2010-01-04 2023-10-03          3461
0   KVUE 2010-01-04 2023-05-03          3356
0   GEHC 2010-01-04 2022-12-14          3261
..   ...        ...        ...           ...
0   TSLA 2010-01-04 2010-06-28           122
0   CBOE 2010-01-04 2010-06-14           112
0    LYB 2010-01-04 2010-04-27            79
0   GNRC 2010-01-04 2010-02-10            27
0   CHTR 2010-01-04 2010-01-04             1

[77 rows x 4 columns]

Summary of missing streaks by length:
count      77.000000
mean     1439.883117
std      1029.878224
min         1.000000
25%       596.000000
50%      1207.000000
75%      2317.000000
max      3581.000000
Name: missing_days, dtype: float64

Number of tickers with gaps > 10 days: 76


In [37]:
# Sort to ensure time order per stock
df = df.sort_values(['Stock', 'Date']).reset_index(drop=True)

# For each stock, keep rows only from its first available Close value
def trim_to_first_valid(group):
    first_valid = group['Close'].first_valid_index()
    if first_valid is not None:
        group = group.loc[first_valid:]
    return group

df_trimmed = df.groupby('Stock', group_keys=False).apply(trim_to_first_valid)

print("Original data shape:", df.shape)
print("Trimmed data shape:", df_trimmed.shape)

# Verify missing Close values are gone
print(df_trimmed['Close'].isna().sum())

Original data shape: (1898322, 8)
Trimmed data shape: (1787451, 8)
0


In [38]:
df_trimmed.drop(columns=['is_missing'], axis=1)

,Date,Open,High,Low,Close,Volume,Stock
0,2010-01-04,19.948872,20.101397,19.783637,19.891676,3815561.0,A
1,2010-01-05,19.834482,19.840836,19.548498,19.675602,4186031.0,A
2,2010-01-06,19.605698,19.701025,19.548500,19.605698,3243779.0,A
3,2010-01-07,19.561211,19.586633,19.383267,19.580278,3095172.0,A
4,2010-01-08,19.472230,19.605689,19.319706,19.573912,3733918.0,A
...,...,...,...,...,...,...,...
1898317,2024-12-24,162.115635,163.442663,161.164935,163.105957,1023600.0,ZTS
1898318,2024-12-26,162.135433,164.185390,161.442211,163.918015,2167200.0,ZTS
1898319,2024-12-27,163.353531,164.918241,161.937378,163.006927,1800100.0,ZTS
1898320,2024-12-30,162.303778,162.462233,159.887392,160.669754,1531400.0,ZTS


In [39]:
output_path = os.path.join(os.getcwd(), '../','data', 'cleaned_data.parquet')
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_trimmed.to_parquet(output_path, index=False)
print("Dataset saved successfully as", output_path)

Dataset saved successfully as d:\DAU\SEM 1\DS605 - Fundamentals of Machine Learning\portfolio_optimization\notebooks\../data\cleaned_data.parquet
